# 逐行学习：从零实现 GPT 模型

本 Notebook 基于《Build a Large Language Model From Scratch》第四章，将代码拆解为可交互的单元格，并添加详细注释。

**学习目标**：
1. 理解 GPT 类模型的整体架构
2. 掌握每个子模块（层归一化、前馈网络、Transformer 块）的作用与实现
3. 学会计算模型参数量与显存占用
4. 实现简单的自回归生成函数

> 建议按顺序执行每个单元格，并尝试修改超参数观察变化。

In [ ]:
# 导入必要的库
import torch
import torch.nn as nn
import tiktoken
from importlib.metadata import version

print("torch version:", version("torch"))
print("tiktoken version:", version("tiktoken"))

## 1. 模型配置 (Config)

我们使用一个字典来存储所有超参数，方便修改和复用。

In [ ]:
GPT_CONFIG_124M = {
    "vocab_size": 50257,      # 词汇表大小（GPT-2 BPE）
    "context_length": 1024,   # 最大上下文长度（位置编码支持）
    "emb_dim": 768,           # 词嵌入维度
    "n_heads": 12,            # 注意力头数
    "n_layers": 12,           # Transformer 块堆叠数量
    "drop_rate": 0.1,         # Dropout 概率
    "qkv_bias": False         # 是否在 QKV 线性层中使用偏置
}

# 打印配置确认
print("模型配置：")
for k, v in GPT_CONFIG_124M.items():
    print(f"  {k}: {v}")

## 2. 层归一化 (LayerNorm)

**作用**：稳定训练，加速收敛。对每个样本的特征维度进行标准化，然后缩放和平移。

**注意**：这里使用了 `unbiased=False`，即除以 `n` 而不是 `n-1`，与原始 GPT-2 权重兼容。

In [ ]:
class LayerNorm(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        self.eps = 1e-5                     # 防止除零的小量
        self.scale = nn.Parameter(torch.ones(emb_dim))   # 可学习的缩放参数
        self.shift = nn.Parameter(torch.zeros(emb_dim))  # 可学习的平移参数

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)          # 沿特征维度计算均值
        var = x.var(dim=-1, keepdim=True, unbiased=False)  # 有偏方差
        norm_x = (x - mean) / torch.sqrt(var + self.eps)    # 标准化
        return self.scale * norm_x + self.shift       # 缩放和平移

# 测试 LayerNorm 的效果
dummy_input = torch.randn(2, 4, 768)   # batch=2, seq_len=4, emb_dim=768
ln = LayerNorm(emb_dim=768)
out_ln = ln(dummy_input)

print(f"输入均值: {dummy_input.mean(dim=-1).mean():.4f}, 方差: {dummy_input.var(dim=-1).mean():.4f}")
print(f"输出均值: {out_ln.mean(dim=-1).mean():.4f}, 方差: {out_ln.var(dim=-1).mean():.4f}")

## 3. GELU 激活函数

GPT 使用 GELU 而非 ReLU。GELU 是平滑的非线性函数，在负半轴仍有小梯度，有助于梯度流动。

下面是近似实现（原始 GPT-2 采用的版本）。

In [ ]:
class GELU(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x):
        return 0.5 * x * (1 + torch.tanh(
            torch.sqrt(torch.tensor(2.0 / torch.pi)) * 
            (x + 0.044715 * torch.pow(x, 3))
        ))

# 可视化 GELU 与 ReLU 对比（可选）
import matplotlib.pyplot as plt
gelu = GELU()
relu = nn.ReLU()
x = torch.linspace(-3, 3, 200)
y_gelu = gelu(x)
y_relu = relu(x)

plt.figure(figsize=(10, 4))
plt.subplot(1,2,1); plt.plot(x, y_gelu); plt.title("GELU"); plt.grid(True)
plt.subplot(1,2,2); plt.plot(x, y_relu); plt.title("ReLU"); plt.grid(True)
plt.show()

## 4. 前馈网络 (FeedForward)

一个简单的两层 MLP，中间维度扩大 4 倍，使用 GELU 激活。

**Shape**：`(batch, num_tokens, emb_dim) -> (batch, num_tokens, emb_dim)`

In [ ]:
class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(cfg["emb_dim"], 4 * cfg["emb_dim"]),  # 升维
            GELU(),                                            # 激活
            nn.Linear(4 * cfg["emb_dim"], cfg["emb_dim"]),   # 降回原维度
        )

    def forward(self, x):
        return self.layers(x)

# 测试前馈网络
ffn = FeedForward(GPT_CONFIG_124M)
sample_x = torch.randn(2, 4, 768)
out_ffn = ffn(sample_x)
print(f"Input shape: {sample_x.shape} -> Output shape: {out_ffn.shape}")

## 5. 短路连接（残差连接）原理演示

残差连接通过 `x = x + layer(x)` 让梯度直接流过，缓解深层网络的梯度消失问题。

下面我们用一个深层网络对比有无残差连接时浅层梯度的大小。

In [ ]:
class DeepNetDemo(nn.Module):
    def __init__(self, layer_sizes, use_shortcut):
        super().__init__()
        self.use_shortcut = use_shortcut
        self.layers = nn.ModuleList([
            nn.Sequential(nn.Linear(layer_sizes[i], layer_sizes[i+1]), GELU())
            for i in range(len(layer_sizes)-1)
        ])
    
    def forward(self, x):
        for layer in self.layers:
            out = layer(x)
            if self.use_shortcut and x.shape == out.shape:
                x = x + out
            else:
                x = out
        return x

def show_gradients(model, x):
    model.zero_grad()
    out = model(x)
    loss = out.mean()
    loss.backward()
    for name, param in model.named_parameters():
        if param.grad is not None and "weight" in name:
            print(f"{name}: gradient mean = {param.grad.abs().mean().item():.6f}")

layer_sizes = [3, 3, 3, 3, 3, 1]
x_test = torch.randn(1, 3)

print("===== 无残差连接 =====")
model_no = DeepNetDemo(layer_sizes, use_shortcut=False)
show_gradients(model_no, x_test)

print("\n===== 有残差连接 =====")
model_yes = DeepNetDemo(layer_sizes, use_shortcut=True)
show_gradients(model_yes, x_test)

## 6. Transformer 块

将多头因果注意力、前馈网络、层归一化和残差连接组合起来。

**注意**：这里使用 **Pre-LayerNorm**（先归一化再进入子层），这是 GPT-2 实际采用的顺序。

In [ ]:
# 注意：需要导入前一章实现的 MultiHeadAttention
# 如果没有 previous_chapters.py，可以使用下面的简化导入（实际从 llms_from_scratch 包获取）
# 为简化，这里假设已经定义了 MultiHeadAttention（支持因果掩码）

# 由于本 notebook 独立运行，我们手动定义一个最小化的 MultiHeadAttention 占位（仅用于理解形状）
# 实际使用时请替换为真正的实现。为了教学，这里直接从官方包导入（如果安装了 llms-from-scratch）
try:
    from previous_chapters import MultiHeadAttention
except ImportError:
    # 如果无法导入，定义一个简单的占位类（仅用于演示形状，不保证正确计算）
    class MultiHeadAttention(nn.Module):
        def __init__(self, d_in, d_out, context_length, num_heads, dropout, qkv_bias):
            super().__init__()
            self.num_heads = num_heads
            self.head_dim = d_out // num_heads
            self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
            self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
            self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
            self.out_proj = nn.Linear(d_out, d_out)
            self.dropout = nn.Dropout(dropout)
        def forward(self, x):
            # 返回形状不变的张量
            return self.out_proj(self.dropout(x))

class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.att = MultiHeadAttention(
            d_in=cfg["emb_dim"],
            d_out=cfg["emb_dim"],
            context_length=cfg["context_length"],
            num_heads=cfg["n_heads"],
            dropout=cfg["drop_rate"],
            qkv_bias=cfg["qkv_bias"]
        )
        self.ff = FeedForward(cfg)
        self.norm1 = LayerNorm(cfg["emb_dim"])
        self.norm2 = LayerNorm(cfg["emb_dim"])
        self.drop_shortcut = nn.Dropout(cfg["drop_rate"])

    def forward(self, x):
        # 注意力部分
        shortcut = x
        x = self.norm1(x)
        x = self.att(x)
        x = self.drop_shortcut(x)
        x = x + shortcut

        # 前馈部分
        shortcut = x
        x = self.norm2(x)
        x = self.ff(x)
        x = self.drop_shortcut(x)
        x = x + shortcut
        return x

# 测试 Transformer Block 的形状
block = TransformerBlock(GPT_CONFIG_124M)
dummy_x = torch.randn(2, 4, 768)
out_block = block(dummy_x)
print(f"Transformer Block 输入形状: {dummy_x.shape}")
print(f"Transformer Block 输出形状: {out_block.shape}")

## 7. 完整的 GPT 模型

将词嵌入、位置嵌入、Dropout、多个 Transformer 块、最终层归一化和输出线性层串联起来。

In [ ]:
class GPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        self.drop_emb = nn.Dropout(cfg["drop_rate"])
        
        self.trf_blocks = nn.Sequential(
            *[TransformerBlock(cfg) for _ in range(cfg["n_layers"])]
        )
        
        self.final_norm = LayerNorm(cfg["emb_dim"])
        self.out_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False)

    def forward(self, in_idx):
        batch_size, seq_len = in_idx.shape
        tok_embeds = self.tok_emb(in_idx)                               # [batch, seq_len, emb_dim]
        pos_embeds = self.pos_emb(torch.arange(seq_len, device=in_idx.device))  # [seq_len, emb_dim]
        x = tok_embeds + pos_embeds                                     # 广播相加
        x = self.drop_emb(x)
        x = self.trf_blocks(x)
        x = self.final_norm(x)
        logits = self.out_head(x)                                       # [batch, seq_len, vocab_size]
        return logits

# 实例化模型
model = GPTModel(GPT_CONFIG_124M)
print(f"模型参数量（未绑定权重）: {sum(p.numel() for p in model.parameters()):,}")

### 权重绑定（Weight Tying）

原始 GPT-2 使用权重绑定：输出层复用词嵌入矩阵。这样可以减少参数量。

下面我们手动绑定并重新计算参数量。

In [ ]:
# 复制模型实例，绑定权重
model_tied = GPTModel(GPT_CONFIG_124M)
model_tied.out_head.weight = model_tied.tok_emb.weight  # 权重共享

# 计算参数量时要去掉重复计数
total_params_tied = sum(p.numel() for name, p in model_tied.named_parameters() if 'out_head.weight' not in name)
print(f"权重绑定后的可训练参数量: {total_params_tied:,}")
print(f"节省的参数: {sum(p.numel() for p in model.out_head.parameters()):,}")

## 8. 文本生成（贪心解码）

虽然模型尚未训练，但我们可以演示生成的过程。

**生成逻辑**：
1. 输入 token IDs
2. 模型输出 logits
3. 取最后一个位置的 logits，应用 softmax 得到概率
4. 贪心选择概率最高的 token ID
5. 将新 token 拼接到输入，重复直到达到指定长度

In [ ]:
def generate_text_simple(model, idx, max_new_tokens, context_size):
    """
    model: GPTModel 实例
    idx: (batch, seq_len) 输入 token 索引
    max_new_tokens: 生成的最大新 token 数
    context_size: 模型支持的最大上下文长度
    """
    for _ in range(max_new_tokens):
        # 如果当前序列超过 context_size，只取最后 context_size 个 token
        idx_cond = idx[:, -context_size:]
        
        with torch.no_grad():
            logits = model(idx_cond)                # (batch, seq_len, vocab_size)
        
        # 只取最后一个时间步的 logits
        logits_last = logits[:, -1, :]              # (batch, vocab_size)
        probas = torch.softmax(logits_last, dim=-1) # 概率分布
        idx_next = torch.argmax(probas, dim=-1, keepdim=True)  # (batch, 1)
        
        # 拼接到原序列
        idx = torch.cat((idx, idx_next), dim=1)
    
    return idx

# 准备输入文本
tokenizer = tiktoken.get_encoding("gpt2")
start_text = "Hello, I am"
input_ids = tokenizer.encode(start_text)
input_tensor = torch.tensor(input_ids).unsqueeze(0)  # (1, seq_len)

model.eval()  # 关闭 dropout
output_ids = generate_text_simple(
    model,
    idx=input_tensor,
    max_new_tokens=6,
    context_size=GPT_CONFIG_124M["context_length"]
)

generated_text = tokenizer.decode(output_ids[0].tolist())
print(f"输入: '{start_text}'")
print(f"生成（未训练，随机）: '{generated_text}'")

## 9. 总结与思考

**你已经学会**：
- GPT 模型的核心组件：嵌入、位置编码、多头注意力、前馈网络、层归一化、残差连接。
- 如何计算模型的参数量及影响内存的关键参数（`emb_dim`, `n_layers`, `vocab_size`）。
- 自回归生成的基本流程（贪心解码）。

**下一步**：
- 训练模型（下一章）
- 加载预训练权重
- 实现更高级的采样方法（temperature、top-k、top-p）

**练习**：
1. 修改 `GPT_CONFIG_124M` 中的 `n_layers` 或 `emb_dim`，观察模型参数量的变化。
2. 尝试将 `generate_text_simple` 中的贪心解码改为按概率随机采样（使用 `torch.multinomial`），观察生成差异。
3. 输出 `model` 的每一层名称，了解模型结构。